# CinderTrace clean rebuild — Stage 4: swath reconciliation and native-data verification

Stage 3 passed: 60 verified L3 files, 372 unique tile/day layers, 10 identical overlaps removed, and 31 days per active product/month. Both disputed months have zero L3 fire-pixel-days. Observation quality varies: December 2023 includes missing-input pixels. Every selected regional pixel had at least one usable daily classification; this does not establish full observing coverage.

This notebook resumes **20261114T163518Z_5b1b1c54**. It reconciles the saved fire-swath and geolocation inventories, checks direct LAADS listings for missing CMR matches, and verifies a small native L2 sample before a larger swath download. It downloads at most five control swaths plus any recovered catalogue omission. Geolocation-file bytes and the full historical archive are not downloaded by this stage.

Native detections and L3 gridded pixel-days are different units. The L3 control totals (9 Terra pixel-days in January 2002; 0 Terra and 4 Aqua in December 2022) must not be compared as identical counts to FIRMS point rows or H3 cell-days.

CPU runtime; VS Code or browser Colab. Run cells in order. No upload widget is required. No zero restoration or production export occurs.


## 1. Install the recorded environment


In [ ]:
%pip -q install earthaccess==0.19.0 requests==2.32.4 pandas==2.2.3 numpy==2.1.3 pyarrow==23.0.1 duckdb==1.3.2 shapely==2.2.0 h3==4.5.0 pyhdf==0.11.7 pyproj==3.8.0 scipy==1.16.3


## 2. Resume the run and verify the previous checkpoint


In [ ]:
from google.colab import drive
from pathlib import Path
import json, hashlib, re, urllib.parse
from datetime import datetime, timezone
import importlib.metadata as metadata
import pandas as pd
from shapely.geometry import shape, Polygon

drive.mount('/content/drive')
RUN_ID = '20261114T163518Z_5b1b1c54'
RUN = Path('/content/drive/MyDrive/cindertrace_clean_rebuild/runs') / RUN_ID
assert RUN.is_dir(), 'The Stage 1 run is not present in this Google Drive account.'
CONFIG = json.loads((RUN/'config.json').read_text())
COLLECTIONS = json.loads((RUN/'inventory/collections.json').read_text())
QUERY_LOG = json.loads((RUN/'inventory/query_log.json').read_text())
expected_versions = json.loads((RUN/'logs/environment.json').read_text())['packages']
actual_versions = {name:metadata.version(name) for name in expected_versions}
assert actual_versions == expected_versions, 'Versions differ. Restart the runtime after installation and rerun.'
STAGE = RUN/'stage2'
STAGE.mkdir(exist_ok=True)
LOCAL = Path('/content/cindertrace_stage2')/RUN_ID
LOCAL.mkdir(parents=True,exist_ok=True)
print('Resumed:',RUN_ID)
print('Pilot months:',CONFIG['pilot_months'])

import numpy as np,requests,time,shutil,uuid
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from pyhdf.SD import SD,SDC
from shapely import contains_xy
from html.parser import HTMLParser

S4=RUN/'stage4';S4.mkdir(exist_ok=True)
META=S4/'inventory';META.mkdir(exist_ok=True)
RAW4=RUN/'raw'/'verified_l2_samples';RAW4.mkdir(parents=True,exist_ok=True)
WORK4=Path('/content/cindertrace_stage4')/RUN_ID;WORK4.mkdir(parents=True,exist_ok=True)
BOUNDARY_BYTES=(STAGE/'Sri_Lanka_pilot_boundary.geojson').read_bytes()
BOUNDARY_SHA256='838db6bf67cb4d21df585d32483c64c9d56556d4fc216564bf4aa982c659382e'
assert hashlib.sha256(BOUNDARY_BYTES).hexdigest()==BOUNDARY_SHA256
REGION=shape(json.loads(BOUNDARY_BYTES)['features'][0]['geometry'])
assert REGION.is_valid
latest=json.loads((RUN/'stage3/latest_analysis.json').read_text())
analysis_relative=Path(latest['path'])
assert not analysis_relative.is_absolute() and '..' not in analysis_relative.parts
PREVIOUS=RUN/analysis_relative
prior_bytes=(PREVIOUS/'report.json').read_bytes()
assert hashlib.sha256(prior_bytes).hexdigest()==latest['report_sha256']
PRIOR=json.loads(prior_bytes)
assert PRIOR['files_verified']==60 and PRIOR['unique_tile_day_rows']==372
assert not PRIOR['missing_tile_day_layers']
assert PRIOR['boundary_sha256']==BOUNDARY_SHA256
print('Stage 3 checkpoint verified:',PREVIOUS)


## 3. Read and compare the saved swath inventories

The saved CMR responses are rechecked against Stage 1 hashes. Pairing uses the satellite and acquisition key (`YYYYDDD.HHMM`), not detection activity. Keep bbox candidates during this comparison: different catalogue footprints can otherwise create a false missing-acquisition claim. Ambiguous multiple processing versions stop the reconciliation.


In [ ]:
def saved_entries(product, month):
    pages = {}
    for query in QUERY_LOG:
        url = urllib.parse.urlsplit(query['url'])
        params = urllib.parse.parse_qs(url.query)
        if not url.path.endswith('/granules.json'):continue
        if params.get('collection_concept_id') != [COLLECTIONS[product]]:continue
        if not params.get('temporal',[''])[0].startswith(month+'-01T'):continue
        if 'snapshot' not in query:
            raise ValueError('This query lacks an immutable snapshot reference; review the Stage 1 notebook version.')
        page = int(params.get('page_num',['1'])[0])
        pages.setdefault(page,{})[query['sha256']] = query
    assert pages and sorted(pages)==list(range(1,max(pages)+1)), f'Missing metadata pages: {product} {month}'
    entries=[]
    for page,variants in sorted(pages.items()):
        assert len(variants)==1, f'Multiple metadata snapshots: {product} {month} page {page}. Review before proceeding.'
        query=next(iter(variants.values()))
        filename=query['snapshot']
        assert Path(filename).name==filename
        raw=(RUN/'inventory'/filename).read_bytes()
        assert hashlib.sha256(raw).hexdigest()==query['sha256'], 'Source metadata hash mismatch.'
        entries.extend(json.loads(raw)['feed']['entry'])
    assert len({e['id'] for e in entries})==len(entries), 'Duplicate granule IDs in inventory.'
    return entries


def acquisition(name):
    match=re.search(r'\.A(\d{7})\.(\d{4})\.',name)
    assert match, f'Unrecognized acquisition name: {name}'
    return match.group(1)+'.'+match.group(2)

def footprint_intersection(entry):
    if not entry.get('polygons'):return None
    hits=[]
    for group in entry['polygons']:
        for text in group:
            xy=list(map(float,text.split()))
            if len(xy)<8 or len(xy)%2:return None
            polygon=Polygon([(xy[i+1],xy[i]) for i in range(0,len(xy),2)])
            if not polygon.is_valid:return None
            hits.append(polygon.intersects(REGION))
    return bool(any(hits))

INVENTORY={};entry_by_key={};pair_summary=[]
for month in CONFIG['pilot_months']:
    for product in ['MOD14','MYD14','MOD03','MYD03']:
        entries=saved_entries(product,month)
        grouped={}
        for entry in entries:
            key=acquisition(entry['producer_granule_id'])
            assert key not in grouped, f'Multiple {product} processing versions for {key}; review before selecting.'
            grouped[key]=entry
        INVENTORY[(month,product)]=grouped
    for sensor,fire,geo in [('Terra','MOD14','MOD03'),('Aqua','MYD14','MYD03')]:
        f=INVENTORY[(month,fire)];g=INVENTORY[(month,geo)]
        pair_summary.append({'month':month,'sensor':sensor,'fire_product':fire,'geolocation_product':geo,
            'fire_bbox_acquisitions':len(f),'geolocation_bbox_acquisitions':len(g),
            'paired_bbox_acquisitions':len(set(f)&set(g)),
            'geolocation_without_fire':sorted(set(g)-set(f)),
            'fire_without_geolocation':sorted(set(f)-set(g))})
pair_table=pd.DataFrame([{k:v for k,v in row.items() if not isinstance(v,list)}|
    {'geolocation_without_fire':len(row['geolocation_without_fire']),
     'fire_without_geolocation':len(row['fire_without_geolocation'])} for row in pair_summary])
display(pair_table)
print('These are catalogue matches, not verified usable regional observations.')


## 4. Reconcile unmatched acquisitions against public sources

An exact CMR acquisition query removes the spatial predicate. If it still finds no fire product, check the corresponding public LAADS day listing. Store exact response bytes and hashes. A directly listed file is a candidate recovery, not a verified observation or proof of download availability.

The known December 21, 2022 Terra acquisition (`2022355.1700`) should be investigated here. Do not hard-code an assumed outage or manufacture a CMR ID for a LAADS-only record.


In [ ]:
public=requests.Session()
public.mount('https://',HTTPAdapter(max_retries=Retry(total=4,backoff_factor=1,status_forcelist=[429,500,502,503,504])))
SOURCE_LOG=[]

def public_snapshot(url,params,label):
    response=public.get(url,params=params,timeout=(30,90));response.raise_for_status()
    raw=response.content;digest=hashlib.sha256(raw).hexdigest()
    path=META/(label+'_'+digest+'.txt')
    if path.exists():assert path.read_bytes()==raw
    else:path.write_bytes(raw)
    SOURCE_LOG.append({'url':response.url,'snapshot':path.name,'sha256':digest})
    return raw,response.url

def cmr_fire_record(entry,product,month,origin='saved_cmr'):
    links=[link['href'] for link in entry.get('links',[]) if not link.get('inherited')
           and link.get('rel','').endswith('/data#')
           and urllib.parse.urlsplit(link.get('href','')).scheme=='https'
           and urllib.parse.urlsplit(link['href']).path.endswith('.hdf')]
    assert len(links)==1, 'Ambiguous CMR science-file link.'
    return {'product':product,'month':month,'name':entry['producer_granule_id'],
        'acquisition':acquisition(entry['producer_granule_id']),'cmr_id':entry['id'],'source':origin,
        'time_start':entry['time_start'],'time_end':entry['time_end'],
        'expected_bytes':round(float(entry['granule_size'])*1048576),
        'download_url':links[0],'footprint_intersects_region':footprint_intersection(entry)}

class Listing(HTMLParser):
    def __init__(self):super().__init__();self.links=[];self.rows=[];self.row=None;self.cell=None
    def handle_starttag(self,tag,attrs):
        if tag=='a':
            href=dict(attrs).get('href')
            if href:self.links.append(href)
        if tag=='tr':self.row=[]
        if tag=='td':self.cell=''
    def handle_data(self,text):
        if self.cell is not None:self.cell+=text
    def handle_endtag(self,tag):
        if tag=='td' and self.row is not None and self.cell is not None:
            self.row.append(' '.join(self.cell.split()));self.cell=None
        if tag=='tr' and self.row is not None:self.rows.append(self.row);self.row=None

RESOLUTIONS=[];RECOVERED=[]
for comparison in pair_summary:
    month=comparison['month'];product=comparison['fire_product'];geo_product=comparison['geolocation_product']
    for key in comparison['geolocation_without_fire']:
        year=key[:4];doy=key[4:7]
        params={'collection_concept_id':COLLECTIONS[product],
            'producer_granule_id':product+'.A'+key+'.061.*',
            'options[producer_granule_id][pattern]':'true','page_size':100}
        raw,_=public_snapshot('https://cmr.earthdata.nasa.gov/search/granules.json',params,'exact_'+product+'_'+key.replace('.','_'))
        matches=json.loads(raw)['feed']['entry']
        assert len(matches)<=1, 'Multiple exact acquisition versions; review before selecting.'
        if matches:
            record=cmr_fire_record(matches[0],product,month,'exact_cmr_query')
            RECOVERED.append(record)
            RESOLUTIONS.append({'month':month,'product':product,'acquisition':key,'status':'found_by_exact_cmr_query','record':record})
            continue
        listing_url=f'https://ladsweb.modaps.eosdis.nasa.gov/api/v2/content/archives/allData/61/{product}/{year}/{doy}/'
        raw,final_url=public_snapshot(listing_url,None,'laads_'+product+'_'+year+doy)
        parser=Listing();parser.feed(raw.decode(errors='replace'))
        pattern=re.compile(re.escape(product+'.A'+key+'.061.')+r'\d+\.hdf')
        found={}
        for href in parser.links:
            url=urllib.parse.urljoin(final_url,href);filename=urllib.parse.urlsplit(url).path.rsplit('/',1)[-1]
            if pattern.fullmatch(filename):found[filename]=url
        assert len(found)<=1, 'Multiple LAADS file versions; review before selecting.'
        if not found:
            RESOLUTIONS.append({'month':month,'product':product,'acquisition':key,'status':'unresolved_not_proven_unrecoverable'})
            continue
        filename,url=next(iter(found.items()))
        size=None
        for row in parser.rows:
            if row and row[0]==filename:
                integers=[int(value.replace(',','')) for value in row[1:] if value.replace(',','').isdigit()]
                if integers:size=integers[0]
        assert size is not None and size>0, 'LAADS file is listed but no unambiguous byte size was parsed; review listing.'
        geo=INVENTORY[(month,geo_product)][key]
        record={'product':product,'month':month,'name':filename[:-4],'acquisition':key,'cmr_id':None,
            'source':'laads_direct_listing','time_start':geo['time_start'],'time_end':geo['time_end'],
            'expected_bytes':size,'download_url':url,'footprint_intersects_region':None}
        RECOVERED.append(record)
        RESOLUTIONS.append({'month':month,'product':product,'acquisition':key,'status':'listed_in_laads_file_not_yet_verified','record':record})

reconciliation={'run_id':RUN_ID,'pair_summary':pair_summary,'resolutions':RESOLUTIONS,'public_sources':SOURCE_LOG,
    'geolocation_files_downloaded':False,'observation_completeness_verified':False}
(S4/'swath_reconciliation.json').write_text(json.dumps(reconciliation,indent=2))
display(pd.DataFrame([{'month':r['month'],'product':r['product'],'acquisition':r['acquisition'],'status':r['status']} for r in RESOLUTIONS]))
print('Reconciliation saved. Remaining mismatches require review; a missing search result is not an outage diagnosis.')


## 5. Define native swath checks and a small resumable downloader

NASA's MODIS fire guide specifies the `FirePix` count and the `FP_` fire-pixel table. `FP_line` and `FP_sample` use zero-based indices. Check the table against the native `fire mask`, retain native confidence/coordinates, and preserve source identity. `FirePix` is a whole-granule count, not a Sri Lanka count.

Source: https://www.earthdata.nasa.gov/s3fs-public/2023-09/MODIS_C6_C6.1_Fire_User_Guide_1.0.pdf

Only a bounded sample is downloaded below. Native point centres are clipped to the same reference polygon; full regional no-fire observation support still requires geolocation and quality accounting.


In [ ]:
class SampleDownloadError(RuntimeError):
    def __init__(self,message,http_status=None):
        super().__init__(message)
        self.http_status=http_status

def read_band(handle,name):
    band=handle.select(name)
    try:return band.get()
    finally:band.endaccess()

def native_table(handle):
    attributes=handle.attributes();names=handle.datasets()
    assert 'FirePix' in attributes and np.asarray(attributes['FirePix']).size==1
    count=int(np.asarray(attributes['FirePix']).item());assert count>=0
    assert 'fire mask' in names and 'algorithm QA' in names, 'Unexpected native MODIS swath schema.'
    mask=read_band(handle,'fire mask')
    assert mask.ndim==2 and mask.shape[1]==1354
    assert tuple(names['algorithm QA'][1])==mask.shape
    assert int(np.isin(mask,[7,8,9]).sum())==count, 'FirePix and native mask disagree.'
    required=['FP_line','FP_sample','FP_latitude','FP_longitude','FP_confidence']
    if count==0:
        return pd.DataFrame({name:pd.Series(dtype='float64') for name in required}),count,mask.shape
    assert all(name in names for name in required), 'Missing native fire-pixel fields.'
    arrays={name:np.asarray(read_band(handle,name)).reshape(-1) for name in required}
    assert all(len(values)==count for values in arrays.values())
    line=arrays['FP_line'].astype(np.int64);sample=arrays['FP_sample'].astype(np.int64)
    assert np.all(arrays['FP_line']==line) and np.all(arrays['FP_sample']==sample)
    assert np.all((line>=0)&(line<mask.shape[0])) and np.all((sample>=0)&(sample<mask.shape[1]))
    assert len(set(zip(line.tolist(),sample.tolist())))==count, 'Duplicate native pixel-table positions.'
    latitude=arrays['FP_latitude'];longitude=arrays['FP_longitude'];confidence=arrays['FP_confidence']
    assert np.isfinite(latitude).all() and np.isfinite(longitude).all() and np.isfinite(confidence).all()
    assert np.all((latitude>=-90)&(latitude<=90)) and np.all((longitude>=-180)&(longitude<=180))
    assert np.all((confidence>=0)&(confidence<=100))
    classes=7+(confidence>=30).astype(np.uint8)+(confidence>=80).astype(np.uint8)
    assert np.array_equal(mask[line,sample],classes), 'Confidence crosswalk and native mask disagree.'
    arrays['fire_mask_class']=classes
    for optional in ['FP_power','FP_T21','FP_T31','FP_SolZenAng']:
        if optional in names:
            values=np.asarray(read_band(handle,optional)).reshape(-1)
            assert len(values)==count
            arrays[optional]=values
    return pd.DataFrame(arrays),count,mask.shape

def swath_file_check(path,record):
    assert abs(path.stat().st_size-record['expected_bytes'])<=2, 'Native file size differs from listing.'
    with path.open('rb') as stream:assert stream.read(4)==b'\x0e\x03\x13\x01', 'Not an HDF4 science file.'
    handle=SD(str(path),SDC.READ)
    try:
        frame,count,dimensions=native_table(handle)
        attributes=handle.attributes()
        geolocation_inputs={key:str(value) for key,value in attributes.items()
            if 'input file' in key.lower() and ('mod03' in key.lower() or 'myd03' in key.lower())}
        metadata={'datasets':{name:list(info[1]) for name,info in handle.datasets().items()},
            'global_fire_pixels':count,'native_mask_shape':list(dimensions),'geolocation_inputs':geolocation_inputs}
    finally:handle.end()
    receipt={'name':record['name'],'cmr_id':record['cmr_id'],'source':record['source'],
        'product':record['product'],'acquisition':record['acquisition'],'bytes':path.stat().st_size,
        'sha256':hashlib.sha256(path.read_bytes()).hexdigest(),'metadata':metadata}
    return receipt,frame

def sample_download(record,session):
    url=urllib.parse.urlsplit(record['download_url'])
    assert url.scheme=='https' and url.hostname in ['data.lpdaac.earthdatacloud.nasa.gov','ladsweb.modaps.eosdis.nasa.gov','data.laadsdaac.earthdatacloud.nasa.gov']
    filename=record['name']+'.hdf';destination=RAW4/filename;receipt_path=RAW4/(record['name']+'.receipt.json')
    if destination.exists():
        receipt,frame=swath_file_check(destination,record)
        if receipt_path.exists():assert json.loads(receipt_path.read_text())['sha256']==receipt['sha256'], 'Cached swath changed.'
        else:receipt_path.write_text(json.dumps(receipt,indent=2))
        return receipt,frame
    local=WORK4/filename;reason='unknown transfer failure';last_http_status=None
    request_headers={'Cache-Control':'no-cache'}
    if url.hostname=='ladsweb.modaps.eosdis.nasa.gov':
        request_headers['X-Requested-With']='XMLHttpRequest'
    for attempt in range(1,5):
        try:
            with session.get(record['download_url'],stream=True,timeout=(30,180),headers=request_headers) as response:
                if response.status_code!=200:raise SampleDownloadError(f'HTTP {response.status_code}',http_status=response.status_code)
                with local.open('wb') as stream:
                    for chunk in response.iter_content(128*1024):
                        if chunk:stream.write(chunk)
            receipt,frame=swath_file_check(local,record)
            temporary=RAW4/(filename+'.partial');shutil.copy2(local,temporary)
            assert hashlib.sha256(temporary.read_bytes()).hexdigest()==receipt['sha256']
            temporary.replace(destination);receipt_path.write_text(json.dumps(receipt,indent=2))
            local.unlink(missing_ok=True)
            return receipt,frame
        except SampleDownloadError as error:
            reason=str(error);last_http_status=error.http_status
            if error.http_status in (401,403):
                local.unlink(missing_ok=True)
                (RAW4/(filename+'.partial')).unlink(missing_ok=True)
                raise SampleDownloadError(f'{filename}: HTTP {error.http_status}; access is unverified, not an observation gap. Check the data provider download token/authorization.',http_status=error.http_status) from None
        except requests.exceptions.RequestException as error:reason=type(error).__name__
        except AssertionError:
            local.unlink(missing_ok=True)
            raise AssertionError(f'Native structure/content check failed for {filename}. Review this file before continuing.') from None
        except OSError as error:reason=type(error).__name__
        local.unlink(missing_ok=True);(RAW4/(filename+'.partial')).unlink(missing_ok=True)
        if attempt<4:
            print(f'Retrying {filename}: {reason}; attempt {attempt+1}/4',flush=True)
            time.sleep(min(2**attempt,8))
    raise SampleDownloadError(f'{filename}: {reason}. Prior samples are cached; review data-provider access or retry the login/download cells.',http_status=last_http_status)


## 6. Verify the native sample and the catalogue-recovered file

Control sample selection is guided by the positive L3 days in January 2002, not by the old detection archive. Stop after finding a swath with a nonempty native fire-pixel table, or after five candidates. The native granule may contain fires outside Sri Lanka; the regional sample count is reported separately.

Then download any recovered fire-product omission from section 4. A 401/403 is recorded as access-denied, without discarding the successful control sample or treating the source as zero. A recovered listing becomes a verified file only after all checks pass. Section 6b provides an optional LAADS Download Token retry when required.


In [ ]:
import earthaccess
auth=earthaccess.login(strategy='interactive',persist=False)
assert auth.authenticated
session=earthaccess.get_requests_https_session()
DAILY_L3=pd.read_parquet(PREVIOUS/'daily_regional_counts.parquet')
positive_days=DAILY_L3[(DAILY_L3.month=='2002-01')&(DAILY_L3['product']=='MOD14A1')&(DAILY_L3.fire_pixel_days>0)].sort_values('fire_pixel_days',ascending=False)
assert len(positive_days), 'The expected positive control has no positive L3 days; review the previous checkpoint.'
day_rank={pd.Timestamp(day).strftime('%Y-%m-%d'):i for i,day in enumerate(positive_days.date)}
candidates=[cmr_fire_record(entry,'MOD14','2002-01') for entry in INVENTORY[('2002-01','MOD14')].values()
            if pd.Timestamp(entry['time_start']).strftime('%Y-%m-%d') in day_rank and footprint_intersection(entry) is not False]
candidates=sorted(candidates,key=lambda record:(day_rank[pd.Timestamp(record['time_start']).strftime('%Y-%m-%d')],record['time_start']))[:5]
assert candidates
sample_rows=[];sample_receipts=[];regional_samples=[];downloaded_names=set();positive_native=False
RECOVERY_ATTEMPTS=[]

def retain_sample(record,receipt,frame,reason):
    inside=contains_xy(REGION,frame.FP_longitude.to_numpy(),frame.FP_latitude.to_numpy()) if len(frame) else np.array([],dtype=bool)
    local=frame.loc[inside].copy()
    local['source_name']=record['name'];local['product']=record['product']
    local['granule_start_utc']=record['time_start'];local['granule_end_utc']=record['time_end']
    local['time_basis']='granule_interval_not_exact_pixel_time'
    regional_samples.append(local)
    sample_receipts.append(receipt)
    sample_rows.append({'reason':reason,'product':record['product'],'acquisition':record['acquisition'],
        'source':record['source'],'global_fire_pixels':receipt['metadata']['global_fire_pixels'],
        'regional_native_detections_in_this_sample':int(inside.sum()),'bytes':receipt['bytes'],
        'file_verified':True})
    downloaded_names.add(record['name'])
    print('Verified native sample:',record['name'])

try:
    for record in candidates:
        receipt,frame=sample_download(record,session)
        retain_sample(record,receipt,frame,'control_schema_verification')
        if receipt['metadata']['global_fire_pixels']>0:
            positive_native=True
            break
    for record in RECOVERED:
        if record['name'] in downloaded_names:continue
        try:
            receipt,frame=sample_download(record,session)
        except SampleDownloadError as error:
            RECOVERY_ATTEMPTS.append({'name':record['name'],'product':record['product'],
                'acquisition':record['acquisition'],'http_status':error.http_status,
                'status':'listed_download_access_denied' if error.http_status in (401,403) else 'download_failed_unverified',
                'file_verified':False})
            print(f'Recovery remains unverified: {record["name"]}; HTTP {error.http_status or "unknown"}. Successful control samples are retained.')
            continue
        retain_sample(record,receipt,frame,'catalogue_omission_file_verification')
        RECOVERY_ATTEMPTS.append({'name':record['name'],'product':record['product'],
            'acquisition':record['acquisition'],'http_status':200,'status':'file_verified','file_verified':True})
finally:session.close()

SAMPLE_TABLE=pd.DataFrame(sample_rows)
display(SAMPLE_TABLE)
if RECOVERY_ATTEMPTS:display(pd.DataFrame(RECOVERY_ATTEMPTS))
print('Nonempty native fire-pixel table verified:',positive_native)
print('Sample counts are not monthly totals. Full regional no-fire coverage is not verified by these samples.')


## 6b. If LAADS returned 401/403: retry with a Download Token

Run section 5 and section 6 before this cell. If every recovered file verified, this cell does nothing.

If LAADS is still access-denied:

1. Sign in at https://urs.earthdata.nasa.gov/.
2. Open **Generate Token** and generate/show an Earthdata Download Token using NASA's interface.
3. Run the cell below and paste the token **only into the hidden prompt**. Do not paste it into code, notebook outputs or chat. Leave the prompt blank to defer this access check and save a clearly partial checkpoint.

The token is used in memory only with the LAADS HTTPS endpoint; it is not written to Drive or the notebook. It is removed from the session headers after the attempt. The request includes the `X-Requested-With` header documented by LAADS.

A 403 alone does not establish its exact cause. If a valid token still gets 403, leave the acquisition access-unverified and review provider authorization/support; do not bypass security or infer a zero month.

NASA references: https://ladsweb.modaps.eosdis.nasa.gov/learn/download-files-using-edl-tokens/ and https://ladsweb.modaps.eosdis.nasa.gov/tools-and-services/api-v2/quick-start-guide/


In [ ]:
import getpass
assert 'RECOVERY_ATTEMPTS' in globals(), 'Run section 6 first.'
laads_pending=[record for record in RECOVERED if record['name'] not in downloaded_names
    and urllib.parse.urlsplit(record['download_url']).hostname=='ladsweb.modaps.eosdis.nasa.gov']
if not laads_pending:
    print('No unverified LAADS file needs a token retry.')
else:
    laads_token=getpass.getpass('Earthdata Download Token (hidden; blank to defer): ').strip()
    if not laads_token:
        print('Token retry deferred. These files remain access-unverified in the checkpoint.')
        del laads_token
    else:
        laads_session=requests.Session()
        laads_session.headers.update({'Authorization':'Bearer '+laads_token,'X-Requested-With':'XMLHttpRequest'})
        try:
            for record in laads_pending:
                # The token is sent only to the approved LAADS HTTPS host.
                target=urllib.parse.urlsplit(record['download_url'])
                assert target.scheme=='https' and target.hostname=='ladsweb.modaps.eosdis.nasa.gov'
                try:
                    receipt,frame=sample_download(record,laads_session)
                except SampleDownloadError as error:
                    RECOVERY_ATTEMPTS.append({'name':record['name'],'product':record['product'],
                        'acquisition':record['acquisition'],'http_status':error.http_status,
                        'status':'download_token_access_denied' if error.http_status in (401,403) else 'download_token_attempt_failed',
                        'file_verified':False})
                    print(f'LAADS retry remains unverified: {record["name"]}; HTTP {error.http_status or "unknown"}.')
                    continue
                retain_sample(record,receipt,frame,'catalogue_omission_file_verified_with_download_token')
                RECOVERY_ATTEMPTS.append({'name':record['name'],'product':record['product'],
                    'acquisition':record['acquisition'],'http_status':200,'status':'file_verified_with_download_token','file_verified':True})
        finally:
            laads_session.headers.pop('Authorization',None)
            laads_session.cookies.clear()
            laads_session.close()
            del laads_token,laads_session
SAMPLE_TABLE=pd.DataFrame(sample_rows)
display(SAMPLE_TABLE)
if RECOVERY_ATTEMPTS:display(pd.DataFrame(RECOVERY_ATTEMPTS))
print('Unverified recovered files:',[record['name'] for record in RECOVERED if record['name'] not in downloaded_names])


## 7. Save the checkpoint


In [ ]:
pending_recovery=[r['name'] for r in RECOVERED if r['name'] not in downloaded_names]
unresolved_inventory=[r for r in RESOLUTIONS if r['status'].startswith('unresolved')]
CHECKPOINT=S4/'checks'/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8])
CHECKPOINT.mkdir(parents=True,exist_ok=False)
SAMPLE_TABLE.to_csv(CHECKPOINT/'native_sample_summary.csv',index=False)
if regional_samples:pd.concat(regional_samples,ignore_index=True).to_parquet(CHECKPOINT/'regional_native_sample.parquet',index=False)
report={'run_id':RUN_ID,'previous_stage3_report_sha256':latest['report_sha256'],'boundary_sha256':BOUNDARY_SHA256,
    'pair_summary':pair_summary,'resolutions':RESOLUTIONS,'public_sources':SOURCE_LOG,
    'sample_summary':sample_rows,'file_receipts':sample_receipts,'positive_native_schema_verified':positive_native,
    'recovery_attempts':RECOVERY_ATTEMPTS,'pending_recovery_files':pending_recovery,
    'recovered_files_verified':not pending_recovery and not unresolved_inventory,
    'stage_status':'checkpoint_saved_recovery_pending' if pending_recovery or unresolved_inventory else 'native_sample_checks_saved',
    'native_parser_sha256':'c60e6d27a57e290627bedd823b38f8e4b90423311083a58351537a4ed4a692be','native_samples_are_monthly_totals':False,
    'full_swath_batch_downloaded':False,'geolocation_files_downloaded':False,
    'observation_completeness_verified':False,'zero_restoration_enabled':False,'production_export_enabled':False}
(CHECKPOINT/'report.json').write_text(json.dumps(report,indent=2,allow_nan=False))
(S4/'latest_check.json').write_text(json.dumps({'path':str(CHECKPOINT.relative_to(RUN)),
    'sha256':hashlib.sha256((CHECKPOINT/'report.json').read_bytes()).hexdigest()},indent=2))
print('CHECKPOINT:',CHECKPOINT)
print('Checkpoint saved. Native schema verified:',positive_native)
print('Recovery files still unverified:',len(pending_recovery))
print('Review access failures before full swath extraction. No unavailable acquisition was counted as zero.')


## Stage 4 checkpoint

Save this executed notebook back to the project. The important outputs are:

1. The fire/geolocation acquisition comparison table.
2. The exact-query/direct-archive resolution table.
3. The verified native sample table and `Nonempty native fire-pixel table verified` status.
4. The printed `CHECKPOINT` path.

After these are reviewed, the next stage will extract the full pilot's native detections with source provenance and reconcile regional observation support. A sample with zero regional detections must never be used to declare a whole month zero. Full swath geolocation/quality and eligibility decisions remain separate work.

The checkpoint may be partial: inspect `stage_status`, `pending_recovery_files` and `recovery_attempts`. A saved checkpoint does not certify successful access or observing completeness.
